# Démo 3 : autograd, le calcul des gradients

Objectif : voir comment PyTorch calcule les gradients, qui disent dans quel sens corriger chaque paramètre ; puis le piège des gradients qui s'accumulent, et comment prédire sans gradient.

**Données** : onze mesures d'un thermomètre dont on ne connaît pas la graduation, avec la température correspondante en degrés Celsius.

**Ce que vous allez voir**

- comment PyTorch calcule seul une dérivée, sur un exemple connu ;
- les gradients de l'erreur du thermomètre, et un pas de descente fait à la main ;
- le piège des gradients qui s'accumulent, et comment prédire sans gradient.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Une dérivée calculée par PyTorch

**PyTorch calcule les dérivées à notre place : c'est l'autograd.**

Pour corriger un paramètre, il faut savoir comment l'erreur varie quand il bouge un peu : c'est la
dérivée, ou **gradient** quand il y a plusieurs paramètres. On vérifie d'abord sur un cas connu : pour
y = x², la pente en x = 3 vaut 2 x 3 = 6.

Le code :
- `torch.tensor(3.0, requires_grad=True)` : `requires_grad=True` demande à PyTorch de suivre tous les calculs
  faits avec `x` ;
- `y = x ** 2` : le calcul, enregistré au passage ;
- `y.backward()` : PyTorch remonte les calculs et calcule la dérivée de `y` par rapport à `x` ;
- `x.grad` : le résultat.

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
print(x.grad)

`tensor(6.)` : PyTorch a calculé la pente de x² en x = 3, sans qu'on lui donne la formule de la dérivée.

**À retenir** : l'autograd dérive n'importe quel calcul fait avec des tenseurs suivis, même un réseau de
millions de paramètres.

## 2. La pente, sur la courbe

**Le gradient est la pente de la courbe en un point.**

Visualiser la dérivée aide à comprendre ce qu'on calcule : une pente positive veut dire que `y`
augmente quand `x` augmente.

Le code :
- `torch.linspace(0, 5, 50)` : 50 valeurs régulièrement espacées entre 0 et 5 ;
- `xs ** 2` : la courbe y = x² ;
- `9 + 6 * (xs - 3)` : la droite qui passe par le point (3, 9) avec la pente 6, la tangente, écrite avec
  la formule de la dérivée (2 x 3 = 6) ;
- `9 + x.grad * (xs - 3)` : la même droite, mais la pente est lue dans `x.grad`, calculée par PyTorch à
  l'étape précédente ;
- `linewidth=6, alpha=0.3` : un trait large et transparent pour la première, afin de voir la seconde par-dessus.

In [ ]:
import matplotlib.pyplot as plt

xs = torch.linspace(0, 5, 50)
plt.plot(xs, xs ** 2, label="y = x²")
plt.plot(xs, 9 + 6 * (xs - 3), linewidth=6, alpha=0.3, label="tangente, pente 6 (formule)")
plt.plot(xs, 9 + x.grad * (xs - 3), "--", label="tangente, pente x.grad (PyTorch)")
plt.legend()
plt.show()

Les deux droites se superposent : le trait large est la tangente écrite avec la formule (pente 6), les
pointillés sont la tangente construite avec `x.grad`. La valeur calculée par PyTorch est donc bien la pente de
la courbe. La droite touche la courbe en x = 3 et suit sa pente : le gradient dit de combien y monte quand
x augmente un peu.

**À retenir** : une pente positive veut dire que diminuer x fait baisser y. C'est ce qu'exploite la descente
de gradient.

## 3. Le graphe de calcul

**Pour calculer les gradients, PyTorch a enregistré le chemin des calculs : le graphe de calcul.**

`backward()` parcourt ce graphe à l'envers, de la sortie vers les entrées. Savoir qu'il existe
explique plusieurs comportements, dont l'accumulation vue plus loin.

Le code :
- `x.is_leaf` : `x` est une **feuille**, un tenseur créé directement par nous ;
- `y.is_leaf` : `y` est issu d'un calcul ;
- `y.grad_fn` : l'opération qui a produit `y`, que `backward()` saura dériver.

In [ ]:
from torchviz import make_dot

make_dot(y, params={"x": x})


In [ ]:
print("x est une feuille :", x.is_leaf)
print("y est une feuille :", y.is_leaf)
print("grad_fn de y      :", y.grad_fn)

`x` est une feuille, `y` non : `y` garde dans `grad_fn` l'opération qui l'a produit, `PowBackward0`, la
puissance. C'est ce chemin que `backward()` remonte pour calculer le gradient.

**À retenir** : les paramètres d'un modèle sont des feuilles ; la loss est au bout du graphe.

## 4. Le thermomètre : deux paramètres

**Retour au thermomètre de la démo 1, avec les paramètres `w` et `b` écrits à la main.**

Dans la démo 1, `nn.Linear` et l'optimiseur cachaient le mécanisme. On le refait à la main pour le
voir.

Le code :
- `celsius`, `x` : les onze mesures, les lectures étant déjà divisées par 10 ;
- `w`, `b` : les deux paramètres, avec `requires_grad=True` pour que PyTorch suive leurs calculs.

In [ ]:
celsius = torch.tensor([0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0])
x = torch.tensor([35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]) / 10
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

Rien à afficher : les onze mesures sont chargées, et `w` et `b` partent de 1 et 0, avec
`requires_grad=True`.

## 5. L'erreur du modèle

**La même mesure d'erreur que dans la démo 1, écrite à la main.**

Le code :
- `w * x + b` : la prédiction du neurone pour les onze lectures ;
- `- celsius` : l'écart avec les vraies températures ;
- `** 2` puis `.mean()` : le carré des écarts, puis leur moyenne.

La fonction ne prend pas d'argument : elle lit `w`, `b`, `x` et `celsius` définis plus haut.

In [ ]:
def erreur():
    return ((w * x + b - celsius) ** 2).mean()


print(erreur().item())

Avec w = 1 et b = 0, l'erreur vaut 80.36 : le modèle est encore loin des mesures.

## 6. Les gradients de l'erreur

**`backward()` calcule en une fois la pente de l'erreur par rapport à `w` et à `b`.**

Chaque gradient dit comment l'erreur change si on augmente un peu le paramètre. Un gradient négatif
veut dire qu'augmenter le paramètre fait baisser l'erreur.

Le code :
- `loss = erreur()` : on calcule l'erreur ;
- `loss.backward()` : on calcule les gradients ;
- `w.grad`, `b.grad` : les deux gradients.

Prédisez : les gradients seront-ils positifs ou négatifs, sachant que le modèle actuel prédit trop bas ?

In [ ]:
loss = erreur()
loss.backward()
print("w.grad =", w.grad.item(), " b.grad =", b.grad.item())
# w.grad.zero_()
# b.grad.zero_()

Les deux gradients sont négatifs (-77.61 pour `w`, -10.64 pour `b`) : augmenter `w` et `b` ferait baisser
l'erreur.

**À retenir** : le gradient donne le sens et l'importance de la correction, pour chaque paramètre.

## 7. Un pas de descente

**On déplace chaque paramètre à l'opposé de son gradient, d'un petit pas : c'est la descente de gradient.**

C'est exactement ce que fait `optimizer.step()` dans la démo 1. Le pas vaut ici 0.01, le learning
rate.

Le code :
- `with torch.no_grad():` : la mise à jour ne doit pas entrer dans le graphe de calcul ; on modifie les
  paramètres, on ne calcule pas une prédiction ;
- `w -= 0.01 * w.grad` : on retire au poids un centième de son gradient ; de même pour `b` ;
- `erreur().item()` : l'erreur après ce pas.

Analogie : descendre une colline dans le brouillard, en faisant un pas dans le sens de la plus forte pente.

In [ ]:
with torch.no_grad():
    w -= 0.01 * w.grad
    b -= 0.01 * b.grad
print(erreur().item())

Après un seul pas, l'erreur tombe à 37.57, deux fois moins qu'avant : les paramètres ont bougé dans le bon
sens. L'entraînement répète ce pas des milliers de fois.

**Essayez** : relancez cette cellule plusieurs fois. L'erreur continue-t-elle de baisser ?

## 8. Piège : les gradients s'accumulent

**Piège classique : `backward()` ajoute les nouveaux gradients aux anciens, au lieu de les remplacer.**

C'est voulu, pour certains usages avancés, mais dans une boucle d'entraînement ordinaire il faut
remettre les gradients à zéro à chaque itération. C'est le rôle de `optimizer.zero_grad()`.

Le code :
- `w.grad.zero_()`, `b.grad.zero_()` : on part de gradients nuls ;
- la boucle appelle `backward()` deux fois, sans rien remettre à zéro entre les deux, et affiche `w.grad`.

Prédisez : que vaudra le second gradient par rapport au premier ?

In [ ]:
w.grad.zero_()
b.grad.zero_()
for appel in range(2):
    erreur().backward()
    print("appel", appel + 1, ": w.grad =", w.grad.item())

Le second appel donne -61.72, exactement le double de -30.86 : `backward()` ajoute le nouveau gradient à
l'ancien au lieu de le remplacer.

**Piège fréquent** : oublier `optimizer.zero_grad()` dans la boucle ; les corrections s'additionnent d'une
itération à l'autre et l'entraînement devient erratique. Le TP1 vous le fera constater.

## 9. Prédire sans gradient

**Pour une simple prédiction, on coupe le suivi des calculs.**

Suivre les calculs coûte de la mémoire et du temps. Pour prédire, on n'en a pas besoin.

Le code :
- `with torch.no_grad():` : aucun calcul du bloc n'est suivi ;
- `.requires_grad` : dit si un résultat est suivi ;
- `.detach()` : une version du tenseur coupée du graphe de calcul, pour un seul tenseur.

In [ ]:
with torch.no_grad():
    print((w * 6.5 + b).requires_grad)
print((w * 6.5 + b).requires_grad)
print((w * 6.5 + b).detach().requires_grad)

Sous `torch.no_grad()`, le résultat ne suit plus les calculs (`False`) ; sans lui, il les suit (`True`) ;
`detach()` coupe ce suivi pour un seul tenseur.

**À retenir** : pour prédire, on n'a pas besoin de gradients ; `torch.no_grad()` économise mémoire et temps.

## À retenir de cette démo

- `requires_grad=True` fait suivre les calculs ; `backward()` calcule les gradients et les range dans `.grad`.
- Le gradient donne le sens de la correction : on déplace chaque paramètre à l'opposé, d'un petit pas.
- Les gradients s'additionnent d'un `backward()` à l'autre : on les remet à zéro à chaque itération. Pour prédire, `torch.no_grad()`.